# Tenfold Tuning Runner

This notebook loads the setup cells from `ten_min_segment_pipeline_v10.4.ipynb`, then runs a grouped 10-fold class-weight x label-smoothing sweep with pooled out-of-fold Severe calibration.

The sweep is executed sequentially with aggressive TensorFlow and plot cleanup between folds and between weight settings to reduce CPU memory buildup.

In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys

DEFAULT_REPO_URL = "https://github.com/zeyneppguler23/CTG_Dissertation.git"
DEFAULT_BRANCH = "helper"
PROJECT_ROOT = Path("/content/CTG_Dissertation")
STAMP_PATH = PROJECT_ROOT / ".colab_requirements_installed"

REPO_URL = os.environ.get("CTG_COLAB_REPO_URL", DEFAULT_REPO_URL).strip()
BRANCH = os.environ.get("CTG_COLAB_REPO_BRANCH", DEFAULT_BRANCH).strip()


def run_command(args, cwd=None, capture_output=False):
    return subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=capture_output,
    )


def remote_branch_exists(repo_url: str, branch: str) -> bool:
    result = subprocess.run(
        ["git", "ls-remote", "--heads", repo_url, branch],
        check=False,
        text=True,
        capture_output=True,
    )
    return result.returncode == 0 and bool(result.stdout.strip())


def requirements_fingerprint(requirements_path: Path) -> str:
    return hashlib.sha256(requirements_path.read_bytes()).hexdigest()


if "google.colab" in sys.modules:
    if not remote_branch_exists(REPO_URL, BRANCH):
        raise RuntimeError(
            "The requested Colab source branch was not found on the remote repository. "
            f"repo={REPO_URL} branch={BRANCH}. "
            "Push the branch first or set CTG_COLAB_REPO_URL / CTG_COLAB_REPO_BRANCH to a valid remote source."
        )

    if not PROJECT_ROOT.exists():
        run_command(
            [
                "git",
                "clone",
                "--branch",
                BRANCH,
                "--single-branch",
                REPO_URL,
                str(PROJECT_ROOT),
            ]
        )
    else:
        print(f"Using existing repo at {PROJECT_ROOT}")
        run_command(["git", "remote", "set-url", "origin", REPO_URL], cwd=PROJECT_ROOT)
        run_command(["git", "fetch", "origin", BRANCH], cwd=PROJECT_ROOT)
        run_command(["git", "checkout", BRANCH], cwd=PROJECT_ROOT)
        run_command(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=PROJECT_ROOT)

    os.environ["CTG_DISSERTATION_ROOT"] = str(PROJECT_ROOT)

    requirements_path = PROJECT_ROOT / "requirements.txt"
    current_fingerprint = requirements_fingerprint(requirements_path)
    installed_fingerprint = STAMP_PATH.read_text(encoding="utf-8").strip() if STAMP_PATH.exists() else ""

    if installed_fingerprint != current_fingerprint:
        run_command(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "--force-reinstall",
                "-r",
                str(requirements_path),
            ]
        )
        STAMP_PATH.write_text(current_fingerprint + "\n", encoding="utf-8")
        raise SystemExit(
            "Dependencies were installed or refreshed for Colab. Restart the runtime now, then run this cell again before running the tuning cell."
        )

    os.chdir(PROJECT_ROOT)
    print(f"Colab repo ready at: {PROJECT_ROOT}")
    print(f"Remote source: {REPO_URL} [{BRANCH}]")
    print("Colab only sees remote code. Unpushed local notebook/script changes are not available there.")
    print("If you just restarted the runtime, run this cell once more and then run the tuning cell.")
    print("Set Runtime -> Change runtime type -> GPU before running the tuning cell.")
else:
    print("Not running in Colab; skipping clone/install bootstrap.")

In [ ]:
from pathlib import Path
import os
import runpy
import sys


def resolve_project_root() -> Path:
    env_root = os.environ.get("CTG_DISSERTATION_ROOT")
    candidate_starts = [Path.cwd().resolve()]
    if env_root:
        candidate_starts.append(Path(env_root).expanduser().resolve())

    candidate_starts.extend(
        [
            Path("/content/CTG_Dissertation"),
            Path("/content/drive/MyDrive/CTG_Dissertation"),
        ]
    )

    seen = set()
    for start in candidate_starts:
        if start in seen:
            continue
        seen.add(start)

        for candidate in [start, *start.parents]:
            runner_candidate = candidate / "src" / "Transfer_Learning" / "run_tenfold_tuning_from_notebook.py"
            if runner_candidate.exists():
                return candidate

    raise FileNotFoundError(
        "Could not locate the CTG_Dissertation project root. "
        "In Colab, clone or copy the repo to /content/CTG_Dissertation, "
        "or set CTG_DISSERTATION_ROOT before running this cell."
    )


PROJECT_ROOT = resolve_project_root()
TRANSFER_LEARNING_DIR = PROJECT_ROOT / "src" / "Transfer_Learning"
os.chdir(TRANSFER_LEARNING_DIR)

if str(TRANSFER_LEARNING_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSFER_LEARNING_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {TRANSFER_LEARNING_DIR}")

runner_path = TRANSFER_LEARNING_DIR / "run_tenfold_tuning_from_notebook.py"
runner_globals = runpy.run_path(str(runner_path))
run_result = runner_globals["run"]()

tenfold_tuning_summary_df = run_result["tenfold_tuning_summary_df"]
tenfold_tuning_summaries = run_result["tenfold_tuning_summaries"]
best_tenfold_run = run_result["best_tenfold_run"]
best_current_weights = run_result["best_current_weights"]
tenfold_tuning_output_dir = run_result["tenfold_tuning_output_dir"]

tenfold_tuning_summary_df

FileNotFoundError: [Errno 2] No such file or directory: '/content/run_tenfold_tuning_from_notebook.py'

In [ ]:
print('Best setting:')
print(f"  Manual class weights: {best_tenfold_run['manual_class_weights']}")
print(f"  Label smoothing: {best_tenfold_run['label_smoothing']:.2f}")
print(f"  OOF Severe boost: {best_tenfold_run['best_oof_severe_boost']:.2f}")
print(f"  Raw balanced accuracy: {best_tenfold_run['raw_balanced_accuracy']:.4f}")
print(f"  Calibrated balanced accuracy: {best_tenfold_run['calibrated_balanced_accuracy']:.4f}")
print(f"  Raw macro F1: {best_tenfold_run['raw_macro_f1']:.4f}")
print(f"  Calibrated macro F1: {best_tenfold_run['calibrated_macro_f1']:.4f}")
print(f"  Outputs: {tenfold_tuning_output_dir}")